---
title: "Private Estimation - cont — Self-study notebook"
subtitle: "Applied Data Privacy"
format:
  html:
    page-layout: full
    toc: false
    toc-depth: 2
    code-tools: true
    code-fold: true
    code-overflow: wrap
    include-in-header:
      text: |
        <style>
        .cell-output-display img, .cell-output-display .plotly-graph-div { max-width: 100%; height: auto; }
        </style>
execute:
  enabled: true
  warning: false
  message: false
jupyter: python3
---

This is the **self-study** companion to the lecture deck: full narrative + all code, meant to be
read and run. For the slide version (code hidden), open the [presentation deck](../../lecture-presentations/private-subgroup-comparisons/presentation.html).


In [ ]:
# Environment setup: pick up local edits to libdpy/ without restarting the kernel.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

try:
    import libdpy
except ImportError:
    %pip install -q "libdpy[notebook] @ git+https://github.com/applied-dp-course/pub_lib.git"
    import libdpy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import display

from libdpy.assignment_specific.private_estimation.utils import (
    DEFAULT_DELTA,
    DEFAULT_SEED,
    gaussian_noise_std,
)
from libdpy.assignment_specific.private_subgroup_comparisons.lecture_figures import (
    build_oracle_ls_failure_artifact,
    build_ptr_failure_artifact,
    build_support_comparison_artifact,
    build_subgroup_sampling_artifact,
    build_typical_oracle_neighbor_pair,
    evaluate_subgroup_accuracy,
    make_subgroup_accuracy_leaderboard_figure,
    make_ptr_failure_probability_figure,
    make_support_comparison_figure,
    make_subgroup_sampling_distribution_figure,
)
from libdpy.visualization.roc_plots import TheoryROCVisualizer
from libdpy.assignment_specific.private_subgroup_comparisons.mechanisms import (
    conceptual_smooth_sensitivity_release,
    global_sensitivity_release,
    noisy_count_sum_release,
    oracle_local_sensitivity_output_law,
    ptr_support_release,
    replacement_ls_bound,
    subgroup_counts,
    subgroup_difference,
)
from libdpy.assignment_specific.private_subgroup_comparisons.witnesses import (
    DEFAULT_EPS_TOTAL,
    DEFAULT_SUPPORT_THRESHOLD,
    DEFAULT_TAU,
    PUBLIC_CLIP_LOWER,
    PUBLIC_CLIP_UPPER,
    PUBLIC_REFERENCE_MEAN,
    PUBLIC_VALUE_UPPER,
    frame_to_arrays,
    prepare_fulton_subgroup_frame,
)

SEED = DEFAULT_SEED
EPS_TOTAL = DEFAULT_EPS_TOTAL
DELTA = DEFAULT_DELTA
rng = np.random.default_rng(SEED)

# PUBLIC CONTRACT (inherited from Lecture 5)
# Clip interval [L, U] and reference mean μ are assumed to have been obtained
# privately, then treated as public for this lecture.
DOLLAR_SCALE = PUBLIC_REFERENCE_MEAN
frame = prepare_fulton_subgroup_frame(seed=SEED)
x, groups = frame_to_arrays(frame)  # default: sex-code comparison
_, latino_groups = frame_to_arrays(frame, group_column="latino_group")

population_df = prepare_fulton_subgroup_frame(n_rows=1_000_000, seed=SEED)
population_x, population_sex_groups = frame_to_arrays(population_df)
_, population_latino_groups = frame_to_arrays(population_df, group_column="latino_group")

LEADERBOARD_N = 1000
LEADERBOARD_N_DATASETS = 40
LEADERBOARD_N_RUNS = 5


def subgroup_value_fn(df):
    return df["x"].to_numpy(dtype=float)


def subgroup_group_fn(df):
    return df["group"].to_numpy()


# Lecture 6 — Private Subgroup Comparisons After Clipping

**Thesis:** After a private step has controlled the salary scale, subgroup analysis is about **denominators and private control flow**, not extreme outliers.

**Public contract:** We start from public quantities that we assume were obtained privately in Lecture 5: a clipping interval and a reference average salary. For this lecture, use the Fulton true 5%-95% income quantile range and clipped population mean as those already-private quantities, then treat them as public post-processing information:

$$
[L,U] = [q_{0.05}, q_{0.95}] = [0, 141{,}450],
\qquad
\mu = \frac{1}{N}\sum_i \mathrm{clip}(y_i,L,U) \approx 33{,}424.
$$

From here on, we work on **mean-scaled clipped salaries**:

$$
x_i = \frac{\mathrm{clip}(y_i,L,U)}{\mu}.
$$

This is not a ratio between groups. It is a unit choice: a subgroup gap $\Delta=0.10$ means the two group averages differ by about $0.10\mu$, or 10% of the public clipped population mean. Dollar-scale estimates are post-processing: multiply by $\mu$.

Because values are clipped before dividing by $\mu$, the public value bound is

$$
0 \le x_i \le B = U/\mu \approx 4.23.
$$

**Subgroup examples:** The main walkthrough uses the dataset's binary sex-code groups (close to 50-50 support). We contrast them with one rarer public boolean cut, **Latino workers versus everyone else** (~5% in Fulton), where denominator stability is less reliable.

**Adjacency:** fixed-size **replacement** witnesses $D' = D - \{z_i\} + \{z'\}$; the replacement may change both value and group membership. In the previous estimation lectures we mostly used the same replacement convention. Add/remove sensitivity has similar denominator behavior, but the exact constants differ.


## S0 — Roadmap

Same rhythm as Lecture 5: define one statistic, inspect ordinary sampling variability, try the tempting oracle-LS proposal, audit why it is invalid, then repair the release with private control flow. After the valid global-sensitivity release, read the signed-error **accuracy leaderboard** on the Fulton sample; conceptual repairs below use different readouts.


## S1 — The statistic

Target: difference of average mean-scaled clipped salaries between two public groups. The main demo compares the two sex-code groups, which have stable support. We also track **Latino workers versus everyone else** as the lecture's single rarer cut (~5%).


### Formal anchor: statistic and denominator floor

For public groups $A,B$, the diagnostic statistic is

$$
\Delta(D)=\frac{1}{n_A}\sum_{i:g_i=A}x_i-\frac{1}{n_B}\sum_{i:g_i=B}x_i,
$$

when both groups are nonempty and $x_i=\mathrm{clip}(y_i,L,U)/\mu$. Private mechanisms below use a total post-processing convention when needed:

$$
\Delta_\tau(D)=\frac{S_A}{\max(n_A,\tau)}-\frac{S_B}{\max(n_B,\tau)}.
$$

This keeps every neighboring input in the domain. Under fixed-size replacement and public bound $0\le x_i\le B$, the conservative local-sensitivity teaching bound is

$$
LS_\Delta(D)\leq B\left(\frac{1}{\max(n_A-1,1)}+\frac{1}{\max(n_B-1,1)}\right).
$$

The constants are intentionally conservative; the lecture uses them to expose the denominator dependence before introducing valid private control flow.


In [ ]:
sex_counts = subgroup_counts(groups)
latino_counts = subgroup_counts(latino_groups)
true_delta = subgroup_difference(x, groups)
latino_delta = subgroup_difference(x, latino_groups)
true_delta_dollars = true_delta * DOLLAR_SCALE
latino_delta_dollars = latino_delta * DOLLAR_SCALE
sex_ls = replacement_ls_bound(sex_counts["A"], sex_counts["B"], value_range=PUBLIC_VALUE_UPPER)
latino_ls = replacement_ls_bound(latino_counts["A"], latino_counts["B"], value_range=PUBLIC_VALUE_UPPER)

def describe_gap(label, delta, dollars):
    print(f"{label} Delta (mean-scaled): {delta:.4f}")
    print(f"{label} Delta (dollars): ${dollars:,.0f}  (post-process: μ·Delta)")

print(f"Public clip [{PUBLIC_CLIP_LOWER:,.0f}, {PUBLIC_CLIP_UPPER:,.0f}]")
print(f"Public reference mean μ = ${PUBLIC_REFERENCE_MEAN:,.0f}; value bound B = U/μ = {PUBLIC_VALUE_UPPER:.3f}")
print(f"Sex-code group counts (stable example): {sex_counts}; conservative LS bound {sex_ls:.4f}")
print(f"Latino vs everyone else counts (rare example): {latino_counts}; conservative LS bound {latino_ls:.4f}")
describe_gap("Sex-code", true_delta, true_delta_dollars)
describe_gap("Latino", latino_delta, latino_delta_dollars)


## S2 — Sampling variability before privacy

Before adding privacy noise, ask how much the **normalized average salary gap** moves under ordinary sampling alone. From the full Fulton population ($N \approx 25{,}766$) we draw $n=1{,}000$ records with replacement, compute the gap, and repeat $2,000$ times. The histogram is the sampling distribution; the vertical line is the full-population target. This baseline is **not a private release** — it separates sampling error from privacy noise.


### Sex-code groups (stable ~50–50 support)

Stable denominators keep the sampling distribution relatively tight. The population sex-code gap is about $0.55$ in mean-scaled units ($\approx \$18{,}500$ after multiplying by $\mu$).


In [ ]:
population_frame = population_df
sex_sampling = build_subgroup_sampling_artifact(
    population_x,
    population_sex_groups,
    sample_size=1000,
    n_samples=2000,
    comparison_label="sex-code groups",
    seed=SEED,
)
fig = make_subgroup_sampling_distribution_figure(
    sex_sampling,
    title="Sampling variability before privacy — sex-code groups",
    xlim=(0.1, 1.0),
)
plt.show()


### Latino workers vs everyone else (~5% minority)

The same resampling design on a rarer cut: minority denominators are smaller, so the histogram is wider even before privacy. The population gap is about $0.46$ mean-scaled units ($\approx \$15{,}400$).


In [ ]:
latino_sampling = build_subgroup_sampling_artifact(
    population_x,
    population_latino_groups,
    sample_size=1000,
    n_samples=2000,
    comparison_label="Latino workers vs everyone else",
    seed=SEED + 1,
)
fig = make_subgroup_sampling_distribution_figure(
    latino_sampling,
    title="Sampling variability before privacy — Latino workers vs everyone else",
    xlim=(0.1, 1.0),
)
plt.show()


## S3 — Global sensitivity is pessimistic

If tiny groups are allowed in the domain, $GS_\Delta \le 2B$, where $B=U/\mu$. This is a valid worst-case bound for the total-function release, but it pays for datasets with nearly empty groups.

For this lecture, the point is the formula, not a plot of a reciprocal curve. The global release is valid; it is usually not the useful baseline for subgroup comparisons with decent support.


In [ ]:
gs_result = global_sensitivity_release(
    x, groups, epsilon=EPS_TOTAL, rng=rng, value_bound=PUBLIC_VALUE_UPPER
)
print(f"Global-sensitivity noise scale for Delta: {gs_result['noise_scale']:.4f}")
print(f"Sex-code local diagnostic scale for Delta: {sex_ls / EPS_TOTAL:.4f}")
print(f"Latino local diagnostic scale for Delta: {latino_ls / EPS_TOTAL:.4f}")
print(f"privacy_status: {gs_result['privacy_status']}")


### Accuracy leaderboard — global sensitivity release

Lecture 5 signed-error decomposition for the valid global-sensitivity Gaussian release on the sex-code walkthrough sample.


In [ ]:
def global_mechanism(values, group_labels, run_rng):
    return global_sensitivity_release(
        values,
        group_labels,
        EPS_TOTAL,
        run_rng,
        value_bound=PUBLIC_VALUE_UPPER,
        delta=DELTA,
    )


global_rows = evaluate_subgroup_accuracy(
    population_df,
    global_mechanism,
    n=LEADERBOARD_N,
    n_datasets=LEADERBOARD_N_DATASETS,
    n_runs=LEADERBOARD_N_RUNS,
    seed=SEED,
    group_fn=subgroup_group_fn,
    value_fn=subgroup_value_fn,
    method="global sensitivity",
    privacy_status="valid",
    epsilon_total=EPS_TOTAL,
)
fig = make_subgroup_accuracy_leaderboard_figure(
    global_rows,
    title="Global sensitivity release on Fulton sex-code sample",
)
plt.show()


## S4 — Temptation: oracle local sensitivity

Bad proposal: compute the conservative replacement local-sensitivity bound on the private dataset, calibrate **Gaussian** noise to that bound at the claimed `(ε, δ)` budget, and release the subgroup gap. This often looks harmless on balanced support because the learned scale barely moves. It is still invalid: the noise standard deviation is a data-dependent function of hidden denominators — the direct analogue of scaling noise with empirical `μ ± kσ` or empirical quantiles in Lecture 5.


First look at a balanced synthetic neighbor pair where one replacement barely changes group support. This is a stability probe, not a privacy proof.


In [ ]:
(
    typical_oracle_x,
    typical_oracle_groups,
    typical_oracle_x_prime,
    typical_oracle_groups_prime,
    typical_oracle_counts_d,
    typical_oracle_counts_d_prime,
) = build_typical_oracle_neighbor_pair(seed=SEED)


The next plot shows the analytical Gaussian ROC for that typical pair. The two output laws use the deterministic subgroup gaps and the **common Gaussian std required for `epsilon = EPS_TOTAL` at `DELTA` on this pair**, not the broken proposal's data-dependent stds. This calibrated analytical ROC shows the `(ε, δ)` bound line; it does not mark a selected-threshold circle because no audit threshold is being selected in this plot.


In [ ]:
typical_oracle_loc, _ = oracle_local_sensitivity_output_law(
    typical_oracle_x,
    typical_oracle_groups,
    EPS_TOTAL,
    DELTA,
    value_bound=PUBLIC_VALUE_UPPER,
)
typical_oracle_loc_prime, _ = oracle_local_sensitivity_output_law(
    typical_oracle_x_prime,
    typical_oracle_groups_prime,
    EPS_TOTAL,
    DELTA,
    value_bound=PUBLIC_VALUE_UPPER,
)
typical_oracle_required_std = gaussian_noise_std(
    abs(typical_oracle_loc - typical_oracle_loc_prime),
    EPS_TOTAL,
    DELTA,
)
display(
    TheoryROCVisualizer(
        "Gaussian",
        scale=typical_oracle_required_std,
        delta=DELTA,
        selectable_distribution=False,
        loc_negative=typical_oracle_loc,
        scale_negative=typical_oracle_required_std,
        loc_positive=typical_oracle_loc_prime,
        scale_positive=typical_oracle_required_std,
        bound_epsilon=EPS_TOTAL,
        show_governing_point=False,
    ).figure()
)


Now move to an engineered sparse-support pair. One replacement removes a row from the smaller group, so the oracle local-sensitivity bound — and therefore the Gaussian noise std — jumps on `D'`.


In [ ]:
sparse_oracle = build_oracle_ls_failure_artifact(seed=SEED)
print(
    f"D counts={sparse_oracle.counts_d}, D' counts={sparse_oracle.counts_d_prime}"
)


This is the primary privacy-failure ROC for the sparse pair. It uses the analytical output laws of the broken oracle-LS proposal: two Gaussians with different deterministic subgroup gaps and different data-dependent noise stds on `D` and `D'`. The thick purple line is the tight `(ε, δ)`-DP bound for this ROC; the open circle marks the governing point where it meets the curve. That tight ε exceeds the claimed `epsilon = EPS_TOTAL` budget.


In [ ]:
sparse_oracle_loc, sparse_oracle_scale = oracle_local_sensitivity_output_law(
    sparse_oracle.x,
    sparse_oracle.groups,
    EPS_TOTAL,
    DELTA,
    value_bound=PUBLIC_VALUE_UPPER,
)
sparse_oracle_loc_prime, sparse_oracle_scale_prime = oracle_local_sensitivity_output_law(
    sparse_oracle.x_prime,
    sparse_oracle.groups_prime,
    EPS_TOTAL,
    DELTA,
    value_bound=PUBLIC_VALUE_UPPER,
)
display(
    TheoryROCVisualizer(
        "Gaussian",
        scale=sparse_oracle_scale,
        delta=DELTA,
        selectable_distribution=False,
        loc_negative=sparse_oracle_loc,
        scale_negative=sparse_oracle_scale,
        loc_positive=sparse_oracle_loc_prime,
        scale_positive=sparse_oracle_scale_prime,
        compute_epsilon=True,
        show_compute_epsilon_toggle=False,
        show_governing_point=True,
    ).figure()
)


## S5 — Valid repair 1: noisy counts and noisy sums

Release four Gaussian queries, then post-process with a public denominator floor $\tau$. Counts are not free, but they are often the honest output.

**Diagnosis addressed:** oracle local sensitivity failed because the noise scale adapted to hidden support. Noisy counts publish support directly (at privacy cost) and keep the release valid by composition.


In [ ]:
count_sum = noisy_count_sum_release(
    x,
    groups,
    EPS_TOTAL / 4,
    EPS_TOTAL / 4,
    EPS_TOTAL / 4,
    EPS_TOTAL / 4,
    DEFAULT_TAU,
    rng,
    value_bound=PUBLIC_VALUE_UPPER,
)
print(count_sum["epsilon"])
print(f"privacy_status: {count_sum['privacy_status']}")
print(f"Private Delta estimate (mean-scaled): {count_sum['estimate']:.4f}")
print(f"Private Delta estimate (dollars): ${count_sum['estimate'] * DOLLAR_SCALE:,.0f}")


## S6 — Conceptual repair 2: Propose-Test-Release

**Diagnosis addressed:** instead of leaking support through the noise scale, PTR certifies stability first. When support is too low, the honest answer is to abstain rather than release a fragile comparison.

The mechanism is defined formally below; `ptr_support_release(...)` implements that definition exactly. We still label the repair **conceptual** because this section states the test-and-release template precisely but does not fully derive the composed `(ε, δ)` guarantee here.


### Formal definition: PTR support certification

Fix public threshold $m \ge 2$, budgets $\varepsilon_{\mathrm{test}}, \varepsilon_{\mathrm{release}} > 0$, and $\delta \in (0,1)$.

**Bad set** (insufficient support):

$$
\mathcal B_m = \{D : \min(n_A, n_B) < m\}.
$$

**Distance to the bad set** (fixed-size replacement; constants may differ by $O(1)$ under other adjacency conventions):

$$
d(D, \mathcal B_m) = \max\{\min(n_A, n_B) - m + 1,\, 0\}.
$$

**Private test.** Publish the noisy distance

$$
\tilde d = d(D, \mathcal B_m) + \mathcal N\!\left(0, \sigma_{\mathrm{test}}^2\right),
\qquad
\sigma_{\mathrm{test}} = \sigma_G(1, \varepsilon_{\mathrm{test}}, \delta),
$$

where $\sigma_G(\Delta, \varepsilon, \delta)$ is the Lecture 5 Gaussian analytic scale (`gaussian_noise_std`) and sensitivity $1$ is how far $d$ can move under one replacement.

Compare $\tilde d$ to the public buffer

$$
b = \frac{\log(2/\delta)}{\varepsilon_{\mathrm{test}}}.
$$

**Decision.**

- If $\tilde d > b$, **accept** and release

$$
\tilde\Delta = \Delta(D) + \mathcal N\!\left(0, \sigma_{\mathrm{release}}^2\right),
\qquad
\sigma_{\mathrm{release}} = \sigma_G(\Delta_{\mathrm{release}}, \varepsilon_{\mathrm{release}}, \delta),
$$

with the public release sensitivity (valid on every accepted dataset under replacement and $0 \le x_i \le B$):

$$
\Delta_{\mathrm{release}} = \frac{2B}{m-1}.
$$

- Otherwise **abstain** (no comparison release).

**Reading.** The test privately estimates distance from $\mathcal B_m$; the release uses a public sensitivity bound that is valid conditional on acceptance, except for the test failure probability. The demo uses `m=DEFAULT_SUPPORT_THRESHOLD`, `eps_test=0.4`, `eps_release=0.6`, and `delta=0.05`.


In [ ]:
ptr = ptr_support_release(
    x,
    groups,
    m=DEFAULT_SUPPORT_THRESHOLD,
    eps_test=0.4,
    eps_release=0.6,
    delta=0.05,
    rng=rng,
    value_bound=PUBLIC_VALUE_UPPER,
)
print(f"status={ptr['status']!r}, accepted={ptr.get('accepted')}, epsilon={ptr['epsilon']}")
if ptr.get("estimate") is not None:
    print(f"released Delta (mean-scaled): {ptr['estimate']:.4f}")

### PTR test failure probability

The private support test abstains when the noisy distance falls below the public buffer. The curve shows the **analytic** probability of that test failure as true $\min(n_A,n_B)$ varies at fixed threshold $m$. The horizontal line marks the chosen failure probability $\delta$; the second line marks $P(\text{release}\mid\text{distance}=0)$ on the bad-set boundary.


In [ ]:
ptr_failure = build_ptr_failure_artifact(
    support_values=np.arange(1, 121, dtype=int),
    ptr_threshold=DEFAULT_SUPPORT_THRESHOLD,
    eps_test=0.4,
    delta=0.05,
)
fig = make_ptr_failure_probability_figure(
    ptr_failure,
    title="PTR support test failure probability vs true minimum support",
    marker_min_count=min(sex_counts.values()),
    marker_label=f"Fulton sex-code sample (min n={min(sex_counts.values())})",
)
plt.show()


### Accuracy leaderboard — PTR support (conceptual)

Conditional releases only; abstentions are dropped from the leaderboard rows.


In [ ]:
def ptr_mechanism(values, group_labels, run_rng):
    return ptr_support_release(
        values,
        group_labels,
        m=DEFAULT_SUPPORT_THRESHOLD,
        eps_test=0.4,
        eps_release=0.6,
        delta=0.05,
        rng=run_rng,
        value_bound=PUBLIC_VALUE_UPPER,
    )


ptr_rows = evaluate_subgroup_accuracy(
    population_df,
    ptr_mechanism,
    n=LEADERBOARD_N,
    n_datasets=LEADERBOARD_N_DATASETS,
    n_runs=LEADERBOARD_N_RUNS,
    seed=SEED + 2,
    group_fn=subgroup_group_fn,
    value_fn=subgroup_value_fn,
    method="PTR support",
    privacy_status="conceptual PTR",
    epsilon_total=EPS_TOTAL,
)
fig = make_subgroup_accuracy_leaderboard_figure(
    ptr_rows,
    title="PTR support release on Fulton sex-code sample",
)
plt.show()


## S7 — Stability repair sketch 3: smooth sensitivity

**Diagnosis addressed:** PTR adapts by abstaining. Smooth sensitivity adapts by adding more noise near unstable regions while asking whether **nearby** datasets are also stable.

The smooth upper bound is defined formally below; `conceptual_smooth_sensitivity_release(...)` implements that Gaussian calibration. We label the repair **conceptual** because this section states the bound precisely but does not prove the full smooth-sensitivity mechanism theorem from the literature.


### Formal definition: smooth sensitivity upper bound

Recall the replacement local-sensitivity teaching bound from S1:

$$
LS_\Delta(D) \le B\left(\frac{1}{\max(n_A-1,1)}+\frac{1}{\max(n_B-1,1)}\right).
$$

Oracle local sensitivity fails because it plugs in the **private** counts. Smooth sensitivity instead upper-bounds how large sensitivity can become on **nearby** datasets.

Let $m(D)=\min(n_A,n_B)$. Under fixed-size replacement, the smaller group count can drop by about one per unit adjacency distance; a conservative symmetric envelope at distance $k$ is

$$
A_k(D) = \frac{2B}{\max(m(D)-k-1,\,1)}.
$$

Fix public parameter $\beta > 0$. The **smooth upper bound** is

$$
S^\beta(D) = \max_{k \ge 0} e^{-\beta k}\, A_k(D).
$$

**Conceptual release.** Calibrate Gaussian noise to $S^\beta(D)$:

$$
\tilde\Delta = \Delta(D) + \mathcal N\!\left(0, \sigma_G(S^\beta(D), \varepsilon, \delta)^2\right),
$$

where $\sigma_G(\Delta, \varepsilon, \delta)$ is the Lecture 5 analytic Gaussian scale (`gaussian_noise_std`).

**Reading.** Smooth sensitivity asks not only whether $D$ is stable, but whether **neighbors** can enter sparse regimes: when $m(D)$ is small, some $A_k(D)$ with small $k$ are large, so $S^\beta(D)$ inflates even if $LS_\Delta(D)$ looks modest on $D$ itself. The helper `smooth_sensitivity_bound(min_count, beta, value_bound=B)` evaluates the max over $k=0,\ldots,m(D)+1$; the demo uses `beta=0.35` and `epsilon=EPS_TOTAL`.


In [ ]:
smooth = conceptual_smooth_sensitivity_release(
    x, groups, epsilon=EPS_TOTAL, beta=0.35, rng=rng, value_bound=PUBLIC_VALUE_UPPER
)
print(f"privacy_status: {smooth['privacy_status']}")
print(f"smooth sensitivity bound: {smooth['smooth_sensitivity']:.4f}")
print(f"released Delta (mean-scaled): {smooth['estimate']:.4f}")

### Accuracy leaderboard — smooth sensitivity (conceptual)


In [ ]:
def smooth_mechanism(values, group_labels, run_rng):
    return conceptual_smooth_sensitivity_release(
        values,
        group_labels,
        epsilon=EPS_TOTAL,
        beta=0.35,
        rng=run_rng,
        value_bound=PUBLIC_VALUE_UPPER,
    )


smooth_rows = evaluate_subgroup_accuracy(
    population_df,
    smooth_mechanism,
    n=LEADERBOARD_N,
    n_datasets=LEADERBOARD_N_DATASETS,
    n_runs=LEADERBOARD_N_RUNS,
    seed=SEED + 3,
    group_fn=subgroup_group_fn,
    value_fn=subgroup_value_fn,
    method="smooth sensitivity",
    privacy_status="conceptual smooth sensitivity",
    epsilon_total=EPS_TOTAL,
)
fig = make_subgroup_accuracy_leaderboard_figure(
    smooth_rows,
    title="Smooth-sensitivity release on Fulton sex-code sample",
)
plt.show()


## S8 — Common support comparison

Fix a PTR threshold $m$ and vary true minimum support $\min(n_A,n_B)$. Compare count/sum error, PTR abstention and conditional error, and smooth-sensitivity error on the same x-axis.

This is the common-footing comparison: which repair is useful when support is large, marginal, or too small?


In [ ]:
support_artifact = build_support_comparison_artifact(
    min_support_values=np.array([3, 5, 8, 12, 20, 30, 50, 80, 120]),
    ptr_threshold=DEFAULT_SUPPORT_THRESHOLD,
    n_large=700,
    n_trials=120,
    eps_total=EPS_TOTAL,
    tau=DEFAULT_TAU,
    seed=SEED,
)
fig = make_support_comparison_figure(support_artifact)
plt.show()


## Part II — Public menu, private search (bridge)

Public data can propose coarsenings and subgroup questions. Private data should only validate support through an explicitly private search primitive. This lecture stops at the bridge; the next lecture handles private search directly.


## Summary and bridge

Analysts rarely ask one subgroup question. They bring a **public menu** of cuts; releasing every count and average is wasteful. Next: private search, report-noisy-max, and preparing models for private learning.